# Actor-Critic (A2C) on CartPole

CSCI 6353 · Topic 38.

An **actor-critic** agent learns two things at once: a policy (the **actor**, $\pi_\theta$)
that chooses actions, and a value function (the **critic**, $V_\phi$) that judges them. The
critic hands the actor a low-variance, per-step signal — the **TD error**
$\delta = r + \gamma V(s') - V(s)$, which is an unbiased estimate of the advantage — curing the
high variance of REINFORCE (Topic 37).

Runs on **CPU**. On Colab: *Runtime → Run all*.

## 1. Setup

In [ ]:
!pip -q install gymnasium torch matplotlib

In [ ]:
import gymnasium as gym
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.distributions import Categorical
import matplotlib.pyplot as plt

learning_rate = 0.0002
gamma = 0.98
n_rollout = 10

## 2. The actor-critic network

A shared body (`fc1`) feeds two heads: `pi` (the actor → action probabilities) and `v` (the
critic → a state value). `train_net` computes the TD error `delta`, uses it as the advantage
for the actor loss, and regresses the critic toward its TD target. The `.detach()` calls are
important: the actor is pushed by the critic's *current* verdict, and the critic trains toward
its *current* target — neither edits the other through its own loss.

In [ ]:
class ActorCritic(nn.Module):
    def __init__(self):
        super().__init__()
        self.data = []
        self.fc1   = nn.Linear(4, 256)   # shared body
        self.fc_pi = nn.Linear(256, 2)   # actor head  -> action logits
        self.fc_v  = nn.Linear(256, 1)   # critic head -> state value
        self.optimizer = optim.Adam(self.parameters(), lr=learning_rate)

    def pi(self, x, softmax_dim=0):
        x = F.relu(self.fc1(x))
        return F.softmax(self.fc_pi(x), dim=softmax_dim)

    def v(self, x):
        x = F.relu(self.fc1(x))
        return self.fc_v(x)

    def put_data(self, transition):
        self.data.append(transition)

    def make_batch(self):
        s, a, r, s2, done = zip(*self.data)
        s  = torch.tensor(s,  dtype=torch.float32)
        a  = torch.tensor(a).unsqueeze(1)
        r  = torch.tensor(r,  dtype=torch.float32).unsqueeze(1) / 100.0   # scale rewards
        s2 = torch.tensor(s2, dtype=torch.float32)
        done_mask = torch.tensor([0.0 if d else 1.0 for d in done]).unsqueeze(1)
        self.data = []
        return s, a, r, s2, done_mask

    def train_net(self):
        s, a, r, s2, done = self.make_batch()
        td_target = r + gamma * self.v(s2) * done       # 0 bootstrap at terminal
        delta = td_target - self.v(s)                   # TD error = advantage estimate
        pi = self.pi(s, softmax_dim=1)
        pi_a = pi.gather(1, a)
        loss = -torch.log(pi_a) * delta.detach() \
               + F.smooth_l1_loss(self.v(s), td_target.detach())
        self.optimizer.zero_grad()
        loss.mean().backward()
        self.optimizer.step()

## 3. Train

The update fires every `n_rollout` steps, not once per episode — the critic lets us learn
continuously. **1,500 episodes** is enough to watch actor-critic reach high scores (a few
minutes on CPU); the lecture uses more.

In [ ]:
EPISODES = 1500

env = gym.make('CartPole-v1')
model = ActorCritic()
returns = []
score = 0.0
for n_epi in range(EPISODES):
    done = False
    s, _ = env.reset()
    s = torch.tensor(s, dtype=torch.float32)
    ep_ret = 0.0
    while not done:
        for _ in range(n_rollout):
            prob = model.pi(s)
            a = Categorical(prob).sample().item()
            s2, r, done, trunc, _ = env.step(a)
            done = done or trunc
            model.put_data((s.numpy(), a, r, s2, done))
            s = torch.tensor(s2, dtype=torch.float32)
            ep_ret += r
            if done:
                break
        model.train_net()
    returns.append(ep_ret)
    score += ep_ret
    if n_epi % 20 == 0 and n_epi != 0:
        print(f'# episode {n_epi:5d}   avg score (last 20): {score/20:6.1f}')
        score = 0.0
env.close()

## 4. The learning curve

Expect a much **faster** climb than REINFORCE — often reaching near-500 within ~1,000–1,200
episodes. Watch for occasional sharp dips after mastery: coupling a moving policy with a
bootstrapping critic can briefly destabilize. Taming that is what PPO (next topics) does.

In [ ]:
import numpy as np
r = np.array(returns); w = 50
ma = np.convolve(r, np.ones(w)/w, mode='valid')
plt.figure(figsize=(8, 4.5))
plt.plot(r, color='#a7f3d0', lw=0.8, label='episode return')
plt.plot(np.arange(w-1, len(r)), ma, color='#16a34a', lw=2.2, label=f'{w}-ep moving avg')
plt.axhline(500, color='#2563eb', ls='--', lw=1.4)
plt.xlabel('episode'); plt.ylabel('return')
plt.title('Actor-Critic (A2C) on CartPole-v1')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Where to go next

- **Compare to REINFORCE.** Run the Topic 37 notebook with the same episode budget and overlay
  the two curves — actor-critic reaches in ~1,000 episodes what REINFORCE does not reach in 3,000.
- **Stabilize it.** The dips you may see motivate **PPO** (Topics 39–40), which limits how far the
  policy moves per update so the actor cannot sprint away from the critic.